In [1]:
import pandas as pd

# Carrega o arquivo Excel
file_path_N90 = '/content/N90_TOTAL_2306.xlsx'
df_N90 = pd.read_excel(file_path_N90)

file_path_N91 = '/content/N91_TOTAL_2306.xlsx'
df_N91 = pd.read_excel(file_path_N91)

file_path_BK = '/content/BACKOFFICE_INFOS_NOVO.xlsx'
df_BK = pd.read_excel(file_path_BK)

# Padronizar nomes das colunas
df_BK.columns = df_BK.columns.str.strip()

# Create a dictionary mapping MUNICIPIO_ID to a list of corresponding ID_LOCALs
locais_municipio = df_BK.groupby('MUNICIPIO_ID')['ID_LOCAL'].apply(list).to_dict()
for municipio, locais in locais_municipio.items():
    locais_municipio[municipio] = list(set(locais))

# Create a dictionary mapping MUNICIPIO_ID to a list of corresponding ID_COORDENACAOs
coordenacoes_municipio = df_BK.groupby('MUNICIPIO_ID')['ID_COORDENACAO'].apply(list).to_dict()
for municipio, locais in coordenacoes_municipio.items():
    coordenacoes_municipio[municipio] = list(set(locais))

# Create a dictionary mapping ID_COORDENACAO to ID_LOCAL for each municipality
coord_local_map = {}
for index, row in df_BK.iterrows():
    municipio = row['MUNICIPIO_ID']
    coordenacao = row['ID_COORDENACAO']
    local = row['ID_LOCAL']
    if municipio not in coord_local_map:
        coord_local_map[municipio] = {}
    coord_local_map[municipio][coordenacao] = local

# Create a reverse mapping from ID_LOCAL to a list of ID_COORDENACAOs for each municipality
local_coord_map = {}
for index, row in df_BK.iterrows():
    municipio = row['MUNICIPIO_ID']
    coordenacao = row['ID_COORDENACAO']
    local = row['ID_LOCAL']
    if municipio not in local_coord_map:
        local_coord_map[municipio] = {}
    if local not in local_coord_map[municipio]:
        local_coord_map[municipio][local] = []
    local_coord_map[municipio][local].append(coordenacao)

# Remove duplicates from the lists of coordinations in local_coord_map
for municipio, locais_info in local_coord_map.items():
    for local, coordinations in locais_info.items():
        local_coord_map[municipio][local] = list(set(coordinations))


# =========================
# DICIONÁRIOS GERADOS AUTOMATICAMENTE
# =========================

# 1. DICIONÁRIO DE MUNICÍPIOS
municipios = (
    df_BK[['MUNICIPIO_ID', 'CIDADE DE APLICACAO']]
    .drop_duplicates()
    .set_index('MUNICIPIO_ID')['CIDADE DE APLICACAO']
    .to_dict()
)

# 2. VAGAS DISPONÍVEIS (POR MUNICÍPIO)
vagas_disponiveis = (
    df_BK
    .groupby('MUNICIPIO_ID')
    .size()
    .mul(30)
    .to_dict()
)

# 3. VAGAS POR LOCAL (APENAS MUNICÍPIOS COM +1 LOCAL)
qtd_locais = df_BK.groupby('MUNICIPIO_ID')['ID_LOCAL'].nunique()
municipios_multilocais = qtd_locais[qtd_locais > 1].index
vagas_disponiveis_local = {}
df_filtrado = df_BK[df_BK['MUNICIPIO_ID'].isin(municipios_multilocais)]
group_local = df_filtrado.groupby(['MUNICIPIO_ID', 'ID_LOCAL']).size().mul(30)
for (municipio, local), valor in group_local.items():
    vagas_disponiveis_local.setdefault(municipio, {})
    vagas_disponiveis_local[municipio][local] = valor

# 4. VAGAS POR COORDENAÇÃO (MUNICIPIO -> LOCAL -> COORD)
vagas_disponiveis_coord_local = {}
group_coord = (
    df_filtrado
    .groupby(['MUNICIPIO_ID', 'ID_LOCAL', 'ID_COORDENACAO'])
    .size()
    .mul(30)
)
for (municipio, local, coord), valor in group_coord.items():
    vagas_disponiveis_coord_local.setdefault(municipio, {})
    vagas_disponiveis_coord_local[municipio].setdefault(local, {})
    vagas_disponiveis_coord_local[municipio][local][coord] = valor


# Criando dois novos dicionários com as mesmas chaves
vagas_disponiveis_manha = {}
vagas_disponiveis_tarde = {}

# Dividindo os valores pela metade e atribuindo aos novos dicionários
for chave, valor in vagas_disponiveis.items():
    vagas_disponiveis_manha[chave] = valor // 2
    vagas_disponiveis_tarde[chave] = valor - (valor // 2) # Ensure total capacity is used

# Combina os dois DataFrames com base na coluna 'CO_INSCRICAO'
df_combinado = pd.merge(df_N90, df_N91, on='CO_INSCRICAO', how='outer')

# Remove registros duplicados com base na coluna CO_INSCRICAO
df_combinado = df_combinado.drop_duplicates(subset='CO_INSCRICAO')

# Adiciona a coluna 'PERIODO_PROVA' com o valor 'TARDE' onde 'CO_PROJETO_y' não é NaN
df_combinado['PERIODO_PROVA'] = df_combinado['CO_PROJETO_y'].apply(lambda x: '16:00' if pd.notna(x) else '')

#print(df_combinado)

# Inicializa um dicionário para contar as vagas preenchidas por local e coordenação
vagas_preenchidas_coord_local_manha = {municipio: {local: {coord: 0 for coord in coords} for local, coords in locais_info.items()} for municipio, locais_info in vagas_disponiveis_coord_local.items()}
vagas_preenchidas_coord_local_tarde = {municipio: {local: {coord: 0 for coord in coords} for local, coords in locais_info.items()} for municipio, locais_info in vagas_disponiveis_coord_local.items()}

# Initialize a dictionary to count filled slots for municipalities with single local (using municipality-wide capacity)
vagas_preenchidas_manha_single_local = {chave: 0 for chave in vagas_disponiveis_manha if chave not in vagas_disponiveis_local}
vagas_preenchidas_tarde_single_local = {chave: 0 for chave in vagas_disponiveis_tarde if chave not in vagas_disponiveis_local}

# Initialize a dictionary to count filled slots for locals with single coordination
vagas_preenchidas_local_single_coord_manha = {municipio: {local: 0 for local in locais} for municipio, locais in vagas_disponiveis_local.items()}
vagas_preenchidas_local_single_coord_tarde = {municipio: {local: 0 for local in locais} for municipio, locais in vagas_disponiveis_local.items()}


# Função para verificar e atribuir o período da prova (simplified as allocation function now determines period)
def atribuir_periodo(row):
    # This function will now only determine the period based on whether a morning slot was assigned
    if row['PERIODO_PROVA'] == '12:00':
        return '12:00'
    else:
        return '16:00'


# Aplica a função em cada linha do dataframe
# First, apply the function to assign ID_COORDENACAO and ID_LOCAL
novo_df = df_combinado[['CO_INSCRICAO', 'NO_PARTICIPANTE', 'CO_MUNICIPIO_PROVA',
                        'SG_UF_MUNICIPIO_PROVA_x','NO_ITEM_ATENDIMENTO','PERIODO_PROVA']].copy()

# Inicializa um dicionário para controlar a alocacao por local e manter a ordem de prioridade
alocacao_local = {}
for municipio, locais_info in vagas_disponiveis_local.items():
    # Sort locals by capacity in descending order
    sorted_locals = sorted(locais_info.items(), key=lambda item: item[1], reverse=True)
    alocacao_local[municipio] = {
        'order': [local for local, capacity in sorted_locals],
        'current_index': {local: 0 for local, capacity in sorted_locals},
        'current_local_index': 0 # To track which local is currently being filled
    }

# Initialize a dictionary to keep track of the current coordination index for each local within a municipality
local_coord_index = {municipio: {local: 0 for local in locais_info} for municipio, locais_info in vagas_disponiveis_local.items()}

# Initialize fallback index dictionaries outside the function to maintain state across rows
if 'fallback_coord_index' not in globals():
    globals()['fallback_coord_index'] = {c: 0 for c in coordenacoes_municipio}
if 'single_local_coord_index' not in globals():
     globals()['single_local_coord_index'] = {c: 0 for c in coordenacoes_municipio if c not in vagas_disponiveis_local}


# Função para atribuir ID_COORDENACAO e ID_LOCAL priorizando locais com maior capacidade and cycling through coordinations within that local, respecting coord capacity
def atribuir_coordenacao_e_local(row):
    co_municipio = row['CO_MUNICIPIO_PROVA']

    assigned_coordination = None
    assigned_local = None
    assigned_period = row['PERIODO_PROVA'] # Get the initial period (16:00 if from N91, or empty)

    # Add debugging prints
    # print(f"Processing CO_MUNICIPIO_PROVA: {co_municipio}")
    # print(f"Initial PERIODO_PROVA: {assigned_period}")

    if co_municipio in alocacao_local:
        # print(f"Municipality {co_municipio} is in alocacao_local.")
        municipio_alloc = alocacao_local[co_municipio]
        local_order = municipio_alloc['order']

        # Find the current local to assign based on the fill order and available capacity
        for local in local_order:
            coordinations_for_local = local_coord_map.get(co_municipio, {}).get(local, [])

            if coordinations_for_local:
                if len(coordinations_for_local) > 1:
                    # Allocate using vagas_disponiveis_coord_local for locals with multiple coordinations
                    total_filled_local = sum(vagas_preenchidas_coord_local_manha[co_municipio].get(local, {}).values()) + sum(vagas_preenchidas_coord_local_tarde[co_municipio].get(local, {}).values())
                    if total_filled_local < vagas_disponiveis_local[co_municipio].get(local, 0): # Check against total local capacity
                         assigned_local = local
                         # print(f"Found available local (multiple coords): {assigned_local}")
                         break
                else:
                    # Allocate using vagas_disponiveis_local for locals with a single coordination
                    total_filled_local_single_coord = vagas_preenchidas_local_single_coord_manha[co_municipio].get(local, 0) + vagas_preenchidas_local_single_coord_tarde[co_municipio].get(local, 0)
                    if total_filled_local_single_coord < vagas_disponiveis_local[co_municipio].get(local, 0):
                         assigned_local = local
                         # print(f"Found available local (single coord): {assigned_local}")
                         break
            # else:
                 # print(f"Local {local} is full. Total filled: {total_filled_local}, Capacity: {vagas_disponiveis_local[co_municipio].get(local, 0)}")


        if assigned_local is not None:
             coordinations_for_local = local_coord_map.get(co_municipio, {}).get(assigned_local, [])

             if coordinations_for_local:
                 if len(coordinations_for_local) > 1:
                     # Allocate using vagas_disponiveis_coord_local
                     start_index = local_coord_index[co_municipio][assigned_local]
                     num_coordinations = len(coordinations_for_local)

                     for i in range(num_coordinations):
                         current_coord_index = (start_index + i) % num_coordinations
                         coord = coordinations_for_local[current_coord_index]

                         if co_municipio in vagas_disponiveis_coord_local and assigned_local in vagas_disponiveis_coord_local[co_municipio] and coord in vagas_disponiveis_coord_local[co_municipio][assigned_local]:
                             coord_capacity_manha = vagas_disponiveis_coord_local[co_municipio][assigned_local][coord] // 2
                             coord_capacity_tarde = vagas_disponiveis_coord_local[co_municipio][assigned_local][coord] - coord_capacity_manha

                             if assigned_period != '16:00' and vagas_preenchidas_coord_local_manha[co_municipio][assigned_local][coord] < coord_capacity_manha:
                                 assigned_coordination = coord
                                 vagas_preenchidas_coord_local_manha[co_municipio][assigned_local][coord] += 1
                                 assigned_period = '12:00'
                                 local_coord_index[co_municipio][assigned_local] = (current_coord_index + 1) % num_coordinations
                                 # print(f"Assigned (multiple coords) to coord {assigned_coordination}, local {assigned_local}, period {assigned_period}")
                                 return assigned_coordination, assigned_local, assigned_period
                             elif vagas_preenchidas_coord_local_tarde[co_municipio][assigned_local][coord] < coord_capacity_tarde:
                                 assigned_coordination = coord
                                 vagas_preenchidas_coord_local_tarde[co_municipio][assigned_local][coord] += 1
                                 assigned_period = '16:00'
                                 local_coord_index[co_municipio][assigned_local] = (current_coord_index + 1) % num_coordinations
                                 # print(f"Assigned (multiple coords) to coord {assigned_coordination}, local {assigned_local}, period {assigned_period}")
                                 return assigned_coordination, assigned_local, assigned_period

                     # If all coordinations within this local are full
                     # print(f"All coordinations in local {assigned_local} are full.")
                     return None, None, '16:00'

                 else:
                     # Allocate using vagas_disponiveis_local for a single coordination
                     coord = coordinations_for_local[0] # Get the single coordination
                     local_capacity_manha = vagas_disponiveis_local[co_municipio].get(assigned_local, 0) // 2
                     local_capacity_tarde = vagas_disponiveis_local[co_municipio].get(assigned_local, 0) - local_capacity_manha

                     if assigned_period != '16:00' and vagas_preenchidas_local_single_coord_manha[co_municipio].get(assigned_local, 0) < local_capacity_manha:
                          assigned_coordination = coord
                          vagas_preenchidas_local_single_coord_manha[co_municipio][assigned_local] += 1
                          assigned_period = '12:00'
                          # print(f"Assigned (single coord, local capacity) to coord {assigned_coordination}, local {assigned_local}, period {assigned_period}")
                          return assigned_coordination, assigned_local, assigned_period
                     elif vagas_preenchidas_local_single_coord_tarde[co_municipio].get(assigned_local, 0) < local_capacity_tarde:
                          assigned_coordination = coord
                          vagas_preenchidas_local_single_coord_tarde[co_municipio][assigned_local] += 1
                          assigned_period = '16:00'
                          # print(f"Assigned (single coord, local capacity) to coord {assigned_coordination}, local {assigned_local}, period {assigned_period}")
                          return assigned_coordination, assigned_local, assigned_period
                     else:
                         # print(f"Local {assigned_local} with single coordination {coord} is full.")
                         return None, None, '16:00'

             else:
                 # If no coordinations found for the assigned local
                 # print(f"No coordinations found for local {assigned_local}.")
                 return None, None, '16:00'


        else:
            # If all prioritized locals are full or no local was assigned, fall back to municipality-wide logic (if applicable)
            # This part remains similar to the previous fallback, but will now only apply to municipalities *not* in vagas_disponiveis_local
            # print(f"All prioritized locals for municipality {co_municipio} are full or none assigned. Falling back to municipality-wide logic.")
            if co_municipio in coordenacoes_municipio and co_municipio not in vagas_disponiveis_local:
                coordinations = coordenacoes_municipio[co_municipio]

                if co_municipio not in single_local_coord_index:
                     single_local_coord_index[co_municipio] = 0

                current_index = single_local_coord_index[co_municipio]
                num_coordinations = len(coordinations)

                for i in range(num_coordinations):
                     current_coord_index = (current_index + i) % num_coordinations
                     assigned_coordination = coordinations[current_coord_index]
                     assigned_local = coord_local_map.get(co_municipio, {}).get(assigned_coordination)

                     if assigned_local is not None:
                         if assigned_period != '16:00' and vagas_preenchidas_manha_single_local.get(co_municipio, 0) < vagas_disponiveis_manha.get(co_municipio, 0):
                              vagas_preenchidas_manha_single_local[co_municipio] += 1
                              assigned_period = '12:00'
                              single_local_coord_index[co_municipio] = (current_coord_index + 1) % num_coordinations
                              # print(f"Assigned (municipality-wide) to coord {assigned_coordination}, local {assigned_local}, period {assigned_period}")
                              return assigned_coordination, assigned_local, assigned_period
                         elif vagas_preenchidas_tarde_single_local.get(co_municipio, 0) < vagas_disponiveis_tarde.get(co_municipio, 0):
                              vagas_preenchidas_tarde_single_local[co_municipio] += 1
                              assigned_period = '16:00'
                              single_local_coord_index[co_municipio] = (current_coord_index + 1) % num_coordinations
                              # print(f"Assigned (municipality-wide) to coord {assigned_coordination}, local {assigned_local}, period {assigned_period}")
                              return assigned_coordination, assigned_local, assigned_period


                # If all coordinations in municipality-wide fallback are full
                # print(f"All coordinations full for municipality-wide logic in municipality {co_municipio}.")
                return None, None, '16:00'

            else:
                # print(f"Municipality {co_municipio} not in either alocacao_local or coordenacoes_municipio for fallback.")
                return None, None, '16:00'


    elif co_municipio in coordenacoes_municipio and co_municipio not in vagas_disponiveis_local:
        # Use municipality-wide logic for municipalities not in vagas_disponiveis_local
        # print(f"Municipality {co_municipio} not in vagas_disponiveis_local. Applying municipality-wide logic.")
        coordinations = coordenacoes_municipio[co_municipio]

        if co_municipio not in single_local_coord_index:
             single_local_coord_index[co_municipio] = 0

        current_index = single_local_coord_index[co_municipio]
        num_coordinations = len(coordinations)

        for i in range(num_coordinations):
            current_coord_index = (current_index + i) % num_coordinations
            assigned_coordination = coordinations[current_coord_index]
            assigned_local = coord_local_map.get(co_municipio, {}).get(assigned_coordination)

            if assigned_local is not None:
                if assigned_period != '16:00' and vagas_preenchidas_manha_single_local.get(co_municipio, 0) < vagas_disponiveis_manha.get(co_municipio, 0):
                     vagas_preenchidas_manha_single_local[co_municipio] += 1
                     assigned_period = '12:00'
                     single_local_coord_index[co_municipio] = (current_coord_index + 1) % num_coordinations
                     # print(f"Assigned (municipality-wide) to coord {assigned_coordination}, local {assigned_local}, period {assigned_period}")
                     return assigned_coordination, assigned_local, assigned_period
                elif vagas_preenchidas_tarde_single_local.get(co_municipio, 0) < vagas_disponiveis_tarde.get(co_municipio, 0):
                     vagas_preenchidas_tarde_single_local[co_municipio] += 1
                     assigned_period = '16:00'
                     single_local_coord_index[co_municipio] = (current_coord_index + 1) % num_coordinations
                     # print(f"Assigned (municipality-wide) to coord {assigned_coordination}, local {assigned_local}, period {assigned_period}")
                     return assigned_coordination, assigned_local, assigned_period


        # If all coordinations are full
        # print(f"All coordinations full for municipality-wide logic in municipality {co_municipio}.")
        return None, None, '16:00'

    else:
        # print(f"Municipality {co_municipio} not in any allocation dictionaries.")
        return None, None, '16:00' # Or some other default value for all three


# Apply the function to each row and create three new columns
novo_df[['ID_COORDENACAO', 'ID_LOCAL', 'PERIODO_PROVA']] = novo_df.apply(atribuir_coordenacao_e_local, axis=1, result_type='expand')

# Ensure ID_COORDENACAO is an integer and remove the extra .0 if it exists
novo_df['ID_COORDENACAO'] = novo_df['ID_COORDENACAO'].apply(lambda x: int(x) if pd.notna(x) else None)
novo_df['ID_LOCAL'] = novo_df['ID_LOCAL'].apply(lambda x: int(x) if pd.notna(x) else None)

# Merge novo_df with df_BK to add CO_COORDENACAO_GRAFICA
novo_df = pd.merge(novo_df, df_BK[['ID_COORDENACAO', 'CO_COORDENACAO_GRAFICA']].drop_duplicates(), on='ID_COORDENACAO', how='left')


# Salva o dataframe 'df_combinado' em um arquivo CSV
#df_combinado.to_csv('df_combinado.csv', index=False)
novo_df.to_csv('novo_df.csv', index=False)

In [ ]:
display(novo_df)

,CO_INSCRICAO,NO_PARTICIPANTE,CO_MUNICIPIO_PROVA,SG_UF_MUNICIPIO_PROVA_x,NO_ITEM_ATENDIMENTO,PERIODO_PROVA,ID_COORDENACAO,ID_LOCAL
0,251120211456605,JAMILE GARCIA MELGAR,4106902,PR,NaN,12:00,47.0,7.0
1,251120211456613,JUSTINE HELLEN CAVALCANTI DE SOUZA,5300108,DF,NaN,12:00,43.0,11.0
2,251120211456621,MICKAEL ANTUNES DE JESUS,4106902,PR,NaN,12:00,47.0,7.0
3,251120211456639,MOHAMED ELSAID AHMED ISMAEL,3550308,SP,NaN,12:00,58.0,48.0
4,251120211456647,BEATRIZ DIAS ROCHA,5208707,GO,NaN,12:00,11.0,23.0
...,...,...,...,...,...,...,...,...
7356,251120211530888,DIANA RODRIGUES LIMA,5300108,DF,NaN,16:00,60.0,50.0
7357,251120211530896,AYDELIAN JEVEY GONZALEZ,2301901,CE,NaN,16:00,55.0,45.0
7358,251120211530938,SANDRA REGINA SILVA TEIXEIRA,3170107,MG,NaN,16:00,NaN,NaN
7359,251120211530953,MICKAELLY LUANNY CORREA CANGIRANA,4115200,PR,NaN,16:00,33.0,32.0
